# 인터뷰 음성 Whisper 변환 및 텍스트 비교 분석

## 목적
- Whisper 모델을 사용하여 m4a 음성 파일을 텍스트로 변환
- 두 번째 발화자(발음이 어눌한 화자)의 음성만 추출하여 분석
- 사람이 타이핑한 텍스트와 Whisper 변환 텍스트의 일치율 분석

## 1. 환경 설정 및 라이브러리 설치

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
# 필요한 라이브러리 설치 (최초 1회)
# !pip install openai-whisper pydub jiwer g2pk librosa soundfile pyannote.audio

In [3]:
# 기존 openai-whisper 대신 transformers 사용
import torch
import librosa
import soundfile as sf
import numpy as np
from pydub import AudioSegment
from jiwer import wer, cer
from g2pk import G2p
import os
import re
import json
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

# Transformers + PEFT (Fine-tuned 모델용)
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/j-i14b110/.local/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA L40S


## 2. 파일 경로 설정 및 로드

In [4]:
# 파일 경로 설정
AUDIO_PATH = "인터뷰 타이핑 원본 음성.m4a"  # 수정 필요
TRANSCRIPT_PATH = "인터뷰 타이핑.txt"  # 수정 필요

# Fine-tuned 모델 경로 (학습 후 저장한 경로)
FINETUNED_MODEL_PATH = "data/들리는json/no7_model_whisper-g2p-final"  # 수정 필요
BASE_MODEL_NAME = "openai/whisper-small"

# 작업 디렉토리
WORK_DIR = "data"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Audio file exists: {os.path.exists(AUDIO_PATH)}")
print(f"Transcript file exists: {os.path.exists(TRANSCRIPT_PATH)}")
print(f"Fine-tuned model exists: {os.path.exists(FINETUNED_MODEL_PATH)}")

Audio file exists: True
Transcript file exists: True
Fine-tuned model exists: True


In [5]:
# 사람이 타이핑한 텍스트 로드
with open(TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    human_transcript = f.read()

print("=== 사람이 타이핑한 텍스트 (앞부분 미리보기) ===")
print(human_transcript[:500])
print(f"\n총 글자 수: {len(human_transcript)}")

=== 사람이 타이핑한 텍스트 (앞부분 미리보기) ===
어어 집에 돌았어요. 아으느 미안해요 제가 시하 주차장에서 시허가 야케서 모태써여
괜차나요 제 자기소개 저는 하구 이천십이니언 삼월달 한국 보러 와써여
네네 한 시보년 정도 돼써요
그 저는 딸이 뚱이 이써요 큰 떨 마ㄱ내 딸 차이 이써서요 어 큰 떨는 지금 이 쓰뭉 요소쌀? ㅁ앙내 따른 열세살 차이가 조금 마나써요
지그는 주부에요. 그냥 집에서 망내 하꾜 꽁부는 거 봐주구 집에서 뭐 주부는거 이리가 하고 이써서요
응 제가 따릉그는 한구거 나이가 조금 있어서 이허리도 안 좋아서 곤부는 거는 한구 곤부는 지금 안 가요
어 타릉거 남펴니 한구 사라미에요
요즈믄 지자는 어려서 그 췌딸 팔딸 그냥 물 씅수 파에 씅 사라미 후빵 파에 씅 사라미 말 드러써요?
후바 발씅 사라미에요
어 따른 부는 또 다른 사라미 다 또가타요 그냥 보통 사라미에요
우리 처으메 와 때 다 한구거 한구거 대화야 이르믈 가장 어려워하는 거에요
그래요 그씨야 쓰는 거 이거 는 거 그러거 대화하는 거 다 그러워여
사시르는 

총 글자 수: 5353


## 3. 오디오 파일 전처리 (m4a → wav 변환)

In [6]:
import os
from pydub import AudioSegment

# ffmpeg/ffprobe 경로 직접 설정
ffmpeg_path = "/home/j-i14b110/.conda/envs/whisper-train/bin/ffmpeg"
ffprobe_path = "/home/j-i14b110/.conda/envs/whisper-train/bin/ffprobe"

AudioSegment.converter = ffmpeg_path
AudioSegment.ffprobe = ffprobe_path

# PATH 환경변수에도 추가
os.environ['PATH'] = "/home/j-i14b110/.conda/envs/whisper-train/bin:" + os.environ['PATH']

print(f"ffmpeg 설정 완료: {ffmpeg_path}")

ffmpeg 설정 완료: /home/j-i14b110/.conda/envs/whisper-train/bin/ffmpeg


In [7]:
# m4a를 wav로 변환 (Whisper는 wav 형식 선호)
wav_path = os.path.join(WORK_DIR, "interview_audio.wav")

# # pydub으로 변환
# audio = AudioSegment.from_file(AUDIO_PATH, format="m4a")
# audio = audio.set_frame_rate(16000)  # 16kHz로 리샘플링
# audio = audio.set_channels(1)  # 모노로 변환
# audio.export(wav_path, format="wav")

# print(f"변환 완료: {wav_path}")
# print(f"오디오 길이: {len(audio) / 1000:.2f}초")
# print(f"샘플레이트: {audio.frame_rate}Hz")
# print(f"채널: {audio.channels}")

## 4. Whisper 모델 로드 및 음성 변환

In [8]:
# === Fine-tuned 모델 로드 ===
print("Loading Fine-tuned Whisper model...")

# 베이스 모델 로드
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)
processor = WhisperProcessor.from_pretrained(BASE_MODEL_NAME)

# LoRA 어댑터 적용
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_PATH)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# 한국어 설정
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="korean", 
    task="transcribe"
)

print(f"Model loaded on: {next(model.parameters()).device}")
print("Fine-tuned model ready!")

Loading Fine-tuned Whisper model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded on: cuda:0
Fine-tuned model ready!


In [9]:
import whisper
import librosa

# === 1단계: openai-whisper로 세그먼트 분리 ===
print("openai-whisper로 세그먼트 분리 중...")
whisper_base = whisper.load_model("small")
result = whisper_base.transcribe(wav_path, language="ko", verbose=False)
base_segments = result["segments"]

print(f"원본 세그먼트 수: {len(base_segments)}")

# === 2단계: 짧은 세그먼트 병합 (최소 5초) ===
def merge_short_segments(segments, min_duration=5.0):
    """짧은 세그먼트를 병합하여 최소 duration 확보"""
    if not segments:
        return []
    
    merged = []
    current = {
        "start": segments[0]["start"],
        "end": segments[0]["end"],
        "text": segments[0]["text"]
    }
    
    for seg in segments[1:]:
        duration = current["end"] - current["start"]
        
        # 현재 세그먼트가 min_duration보다 짧으면 병합
        if duration < min_duration:
            current["end"] = seg["end"]
            current["text"] += " " + seg["text"]
        else:
            merged.append(current)
            current = {
                "start": seg["start"],
                "end": seg["end"],
                "text": seg["text"]
            }
    
    # 마지막 세그먼트 추가
    merged.append(current)
    
    return merged

merged_segments = merge_short_segments(base_segments, min_duration=5.0)
print(f"병합 후 세그먼트 수: {len(merged_segments)}")

# === 3단계: Fine-tuned 모델로 변환 ===
print("\nFine-tuned 모델로 변환 중...")
print("=" * 60)

audio_full, sr = librosa.load(wav_path, sr=16000)

transcripts = []
segments_data = []

forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="korean", 
    task="transcribe"
)

for i, seg in enumerate(merged_segments):
    start_sample = int(seg["start"] * 16000)
    end_sample = int(seg["end"] * 16000)
    
    chunk = audio_full[start_sample:end_sample]
    
    # 너무 짧으면 스킵 (1초 미만)
    if len(chunk) < 16000:
        continue
    
    input_features = processor(
        chunk,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(model.device)
    
    with torch.no_grad():
        predicted_ids = model.generate(
            input_features=input_features,
            max_length=448,
            num_beams=5,
            forced_decoder_ids=forced_decoder_ids,
            
            # 🔥 환각 방지 파라미터
            no_repeat_ngram_size=3,
            repetition_penalty=1.5,
            early_stopping=True,
        )
    
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    
    # 🔥 후처리: 반복 패턴 제거
    import re
    text = re.sub(r'(.)\1{10,}', r'\1', text)  # 같은 글자 10번+ 반복 제거
    text = re.sub(r'(네\s*){5,}', '네 ', text)
    text = re.sub(r'(고맙\s*){3,}', '고맙 ', text)
    text = re.sub(r'MBC 뉴스.*?입니다\.?', '', text)
    text = re.sub(r'구독과 좋아요.*', '', text)
    
    transcripts.append(text)
    segments_data.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": text.strip()
    })
    
    print(f"[{seg['start']:.1f}s - {seg['end']:.1f}s] {text[:70]}...")

whisper_transcript = " ".join(transcripts)

print("\n" + "=" * 60)
print(f"총 변환 세그먼트: {len(segments_data)}")
print(f"\n=== Fine-tuned Whisper 변환 결과 (앞부분) ===")
print(whisper_transcript[:500])

openai-whisper로 세그먼트 분리 중...


100%|██████████| 154438/154438 [00:42<00:00, 3659.20frames/s]


원본 세그먼트 수: 1013
병합 후 세그먼트 수: 252

Fine-tuned 모델로 변환 중...


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

[0.0s - 6.4s]  네 안녕하세요 지금 집에 계시는 건가요? 다음에 집에 돌았어요 미안해요 제가...
[6.4s - 14.1s]  네, 네. 가사 치아쥬 사장에서 신호가 약해서 못했어요. 만나서 반갑습니다....
[14.1s - 20.7s]  일단 저희는 사피에 다니고 있는 학생들이고 앞서 말했듯이...
[20.7s - 29.3s]  이제 다문화 가정분들이 한국어를 배울 때 경험하는 어려움하고 그런 걸 좀 도와주기 위해서...
[29.3s - 34.5s]  서비스를 개발하고 있는데 구체적으로 이제 그런 게 어떤 게 될까 해서...
[34.5s - 42.1s]  이렇게 인터뷰 전화를 드렸습니다 일단 자기소개 질문 좀 하고 싶은데...
[42.1s - 47.1s]  괜찮으실까요? 괜찮아요 제 자기소개 저는...
[47.1s - 52.5s]  한국이 제 12년 3몇 달 한국으로 왔어요...
[52.5s - 58.3s]  2016년이요. 그 한 10년 정도 됐어요. 그렇죠....
[60.3s - 65.4s]  저는 딸이 똥이 있어요 지금 달 만에 달 차이 있어서요...
[65.4s - 71.5s]  네. 캔달은 지금 26살?...
[71.5s - 77.3s]  언니들은 13살 차이가 조금 많았어요. 끊었네요....
[77.3s - 86.2s]  아, 지금은 주부예요. 그냥 집에서 막내 학교 공부는 거 봐주고...
[86.2s - 95.5s]  집에서 주부는 거 일회가 하고 있었어요. 제가 따로는 한국어 나이가 조금 있어서...
[95.5s - 101.7s]  피어리도 안 좋아서 한국 공부는 지금 안 가요....
[101.7s - 106.7s]  아 네 다른 그 남편이 한국 사람이에요...
[106.7s - 113.8s]  네, 네. 요즘은 지창을 어울려서. 네....
[113.8s - 119.5s]  나잇다, 그녀물 생소排송살라미. 구팡排송살아미....
[119.5s - 124.8s]  잘 들었어요? 네네네. 음.. 그방 발성 사람이에요....
[124.8s 

In [10]:
# 세그먼트별 결과 확인 (타임스탬프 포함)
print("=== 세그먼트별 변환 결과 ===")
for i, seg in enumerate(segments_data[:20]):  # 처음 20개만 표시
    start = seg["start"]
    end = seg["end"]
    text = seg["text"]
    print(f"[{start:.2f}s - {end:.2f}s] {text}")

=== 세그먼트별 변환 결과 ===
[0.00s - 6.40s] 네 안녕하세요 지금 집에 계시는 건가요? 다음에 집에 돌았어요 미안해요 제가
[6.40s - 14.10s] 네, 네. 가사 치아쥬 사장에서 신호가 약해서 못했어요. 만나서 반갑습니다.
[14.10s - 20.70s] 일단 저희는 사피에 다니고 있는 학생들이고 앞서 말했듯이
[20.70s - 29.30s] 이제 다문화 가정분들이 한국어를 배울 때 경험하는 어려움하고 그런 걸 좀 도와주기 위해서
[29.30s - 34.50s] 서비스를 개발하고 있는데 구체적으로 이제 그런 게 어떤 게 될까 해서
[34.50s - 42.10s] 이렇게 인터뷰 전화를 드렸습니다 일단 자기소개 질문 좀 하고 싶은데
[42.10s - 47.10s] 괜찮으실까요? 괜찮아요 제 자기소개 저는
[47.10s - 52.50s] 한국이 제 12년 3몇 달 한국으로 왔어요
[52.50s - 58.30s] 2016년이요. 그 한 10년 정도 됐어요. 그렇죠.
[60.30s - 65.40s] 저는 딸이 똥이 있어요 지금 달 만에 달 차이 있어서요
[65.40s - 71.50s] 네. 캔달은 지금 26살?
[71.50s - 77.30s] 언니들은 13살 차이가 조금 많았어요. 끊었네요.
[77.30s - 86.20s] 아, 지금은 주부예요. 그냥 집에서 막내 학교 공부는 거 봐주고
[86.20s - 95.50s] 집에서 주부는 거 일회가 하고 있었어요. 제가 따로는 한국어 나이가 조금 있어서
[95.50s - 101.70s] 피어리도 안 좋아서 한국 공부는 지금 안 가요.
[101.70s - 106.70s] 아 네 다른 그 남편이 한국 사람이에요
[106.70s - 113.80s] 네, 네. 요즘은 지창을 어울려서. 네.
[113.80s - 119.50s] 나잇다, 그녀물 생소排송살라미. 구팡排송살아미.
[119.50s - 124.80s] 잘 들었어요? 네네네. 음.. 그방 발성 사람이에요.
[124.80s - 130.30s] 아 다른 사

## 5. 화자 분리 (Speaker Diarization) - 두 번째 발화자 추출

두 명의 발화자가 있으며, 두 번째 등장하는 발화자(발음이 어눌한 화자)만 추출해야 합니다.

In [11]:
import pyannote.audio
print(pyannote.audio.__version__)

4.0.3


In [11]:
# 세그먼트 저장 (segments_data 사용)
import json

segments_file = os.path.join(WORK_DIR, "whisper_segments7.json")
with open(segments_file, 'w', encoding='utf-8') as f:
    json.dump(segments_data, f, ensure_ascii=False, indent=2)
    
print(f"세그먼트 저장 완료: {segments_file}")
print(f"총 세그먼트 수: {len(segments_data)}")

세그먼트 저장 완료: data/whisper_segments7.json
총 세그먼트 수: 252


## 6. 텍스트 전처리 함수

In [21]:
# 사람이 타이핑한 텍스트 로드
TRANSCRIPT_PATH = "인터뷰 타이핑.txt"

with open(TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    human_transcript = f.read()

print("=== 사람이 타이핑한 텍스트 (앞부분 미리보기) ===")
print(human_transcript[:500])
print(f"\n총 글자 수: {len(human_transcript)}")

=== 사람이 타이핑한 텍스트 (앞부분 미리보기) ===
어어 집에 돌았어요. 아으느 미안해요 제가 시하 주차장에서 시허가 야케서 모태써여
괜차나요 제 자기소개 저는 하구 이천십이니언 삼월달 한국 보러 와써여
네네 한 시보년 정도 돼써요
그 저는 딸이 뚱이 이써요 큰 떨 마ㄱ내 딸 차이 이써서요 어 큰 떨는 지금 이 쓰뭉 요소쌀? ㅁ앙내 따른 열세살 차이가 조금 마나써요
지그는 주부에요. 그냥 집에서 망내 하꾜 꽁부는 거 봐주구 집에서 뭐 주부는거 이리가 하고 이써서요
응 제가 따릉그는 한구거 나이가 조금 있어서 이허리도 안 좋아서 곤부는 거는 한구 곤부는 지금 안 가요
어 타릉거 남펴니 한구 사라미에요
요즈믄 지자는 어려서 그 췌딸 팔딸 그냥 물 씅수 파에 씅 사라미 후빵 파에 씅 사라미 말 드러써요?
후바 발씅 사라미에요
어 따른 부는 또 다른 사라미 다 또가타요 그냥 보통 사라미에요
우리 처으메 와 때 다 한구거 한구거 대화야 이르믈 가장 어려워하는 거에요
그래요 그씨야 쓰는 거 이거 는 거 그러거 대화하는 거 다 그러워여
사시르는 

총 글자 수: 5353


In [22]:
def preprocess_text(text):
    """텍스트 전처리: 비교를 위한 정규화"""
    # 소문자 변환 (영문이 섞여 있을 경우)
    text = text.lower()
    
    # 특수문자 및 구두점 제거 (한글, 숫자, 공백만 유지)
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣ0-9\s]', '', text)
    
    # 연속 공백을 단일 공백으로
    text = re.sub(r'\s+', ' ', text)
    
    # 앞뒤 공백 제거
    text = text.strip()
    
    return text

# 전처리 적용
human_clean = preprocess_text(human_transcript)
whisper_clean = preprocess_text(whisper_transcript)

print("=== 전처리된 사람 타이핑 텍스트 (앞부분) ===")
print(human_clean[:300])
print("\n=== 전처리된 Whisper 텍스트 (앞부분) ===")
print(whisper_clean[:300])

=== 전처리된 사람 타이핑 텍스트 (앞부분) ===
어어 집에 돌았어요 아으느 미안해요 제가 시하 주차장에서 시허가 야케서 모태써여 괜차나요 제 자기소개 저는 하구 이천십이니언 삼월달 한국 보러 와써여 네네 한 시보년 정도 돼써요 그 저는 딸이 뚱이 이써요 큰 떨 마ㄱ내 딸 차이 이써서요 어 큰 떨는 지금 이 쓰뭉 요소쌀 ㅁ앙내 따른 열세살 차이가 조금 마나써요 지그는 주부에요 그냥 집에서 망내 하꾜 꽁부는 거 봐주구 집에서 뭐 주부는거 이리가 하고 이써서요 응 제가 따릉그는 한구거 나이가 조금 있어서 이허리도 안 좋아서 곤부는 거는 한구 곤부는 지금 안 가요 어 타릉거 남펴니 한구

=== 전처리된 Whisper 텍스트 (앞부분) ===
네 안녕하세요 지금 집에 계시는 건가요 손을 집에 돌았어요 미안해요 제가 네 네 네 가사 치아트 타자에서 씨너가 약해서 못했죠 네 만나서 반갑습니다 네 일단 저희는 사피에 다니고 있는 학생들이고 앞서 말했듯이 이제 다문화 가정분들이 한국어를 배울 때 경험하는 어려움하고 그런 걸 좀 도와주기 위해서 서비스를 개발하고 있는데 구체적으로 이제 그런 게 어떤 게 될까 해서 이렇게 인터뷰 전화를 드렸습니다 에잇 일단 자기소개 질문 좀 하고 싶은데 괜찮으실까요 괜찮아요 제 자기소개 저는 한국이 쟨 12년 3몰 딸 한국으로 왔어요 2017년이요


## 7. 일치율 분석 - 다양한 메트릭

In [ ]:
def calculate_similarity_metrics(reference, hypothesis):
    """다양한 유사도/일치율 메트릭 계산"""
    results = {}
    
    # 1. Character Error Rate (CER) - 문자 단위 오류율
    cer_score = cer(reference, hypothesis)
    results['CER'] = cer_score
    results['Character_Accuracy'] = max(0, 1 - cer_score) * 100
    
    # 2. Word Error Rate (WER) - 단어 단위 오류율
    wer_score = wer(reference, hypothesis)
    results['WER'] = wer_score
    results['Word_Accuracy'] = max(0, 1 - wer_score) * 100
    
    # 3. Sequence Matcher - 전체 시퀀스 유사도
    seq_ratio = SequenceMatcher(None, reference, hypothesis).ratio()
    results['Sequence_Similarity'] = seq_ratio * 100
    
    # 4. 길이 비교
    results['Reference_Length'] = len(reference)
    results['Hypothesis_Length'] = len(hypothesis)
    results['Length_Ratio'] = len(hypothesis) / len(reference) * 100 if len(reference) > 0 else 0
    
    return results

# 전체 텍스트 비교
metrics = calculate_similarity_metrics(human_clean, whisper_clean)

print("="*60)
print("전체 텍스트 비교 결과")
print("="*60)
print(f"Character Error Rate (CER): {metrics['CER']:.4f} ({metrics['CER']*100:.2f}%)")
print(f"Character Accuracy: {metrics['Character_Accuracy']:.2f}%")
print(f"\nWord Error Rate (WER): {metrics['WER']:.4f} ({metrics['WER']*100:.2f}%)")
print(f"Word Accuracy: {metrics['Word_Accuracy']:.2f}%")
print(f"\nSequence Similarity: {metrics['Sequence_Similarity']:.2f}%")
print(f"\nReference (Human) Length: {metrics['Reference_Length']} characters")
print(f"Hypothesis (Whisper) Length: {metrics['Hypothesis_Length']} characters")
print(f"Length Ratio: {metrics['Length_Ratio']:.2f}%")

In [ ]:
import re
from jiwer import wer, cer
from difflib import SequenceMatcher

# 파일 경로 설정
HUMAN_TRANSCRIPT_PATH = "인터뷰 타이핑 - 복사본.txt"  # 사람이 타이핑한 파일
WHISPER_TRANSCRIPT_PATH = "fine tuned 변환.txt"  # Whisper 변환 결과 (2번 화자만 정리한 파일명으로 수정)

# 텍스트 파일 로드
with open(HUMAN_TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    human_transcript = f.read()

with open(WHISPER_TRANSCRIPT_PATH, 'r', encoding='utf-8') as f:
    whisper_transcript = f.read()

print(f"사람 타이핑 글자 수: {len(human_transcript)}")
print(f"Whisper 변환 글자 수: {len(whisper_transcript)}")

# 전처리 함수
def preprocess_text(text):
    """텍스트 전처리: 비교를 위한 정규화"""
    text = text.lower()
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣ0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

# 전처리 적용
human_clean = preprocess_text(human_transcript)
whisper_clean = preprocess_text(whisper_transcript)

print("\n=== 전처리된 사람 타이핑 (앞부분) ===")
print(human_clean[:300])
print("\n=== 전처리된 Whisper 텍스트 (앞부분) ===")
print(whisper_clean[:300])

# 일치율 분석 함수
def calculate_similarity_metrics(reference, hypothesis):
    """다양한 유사도/일치율 메트릭 계산"""
    results = {}
    
    # 1. Character Error Rate (CER)
    cer_score = cer(reference, hypothesis)
    results['CER'] = cer_score
    results['Character_Accuracy'] = max(0, 1 - cer_score) * 100
    
    # 2. Word Error Rate (WER)
    wer_score = wer(reference, hypothesis)
    results['WER'] = wer_score
    results['Word_Accuracy'] = max(0, 1 - wer_score) * 100
    
    # 3. Sequence Similarity
    seq_ratio = SequenceMatcher(None, reference, hypothesis).ratio()
    results['Sequence_Similarity'] = seq_ratio * 100
    
    # 4. 길이 비교
    results['Reference_Length'] = len(reference)
    results['Hypothesis_Length'] = len(hypothesis)
    results['Length_Ratio'] = len(hypothesis) / len(reference) * 100 if len(reference) > 0 else 0
    
    return results

# 분석 실행
metrics = calculate_similarity_metrics(human_clean, whisper_clean)

# 결과 출력
print("\n" + "="*60)
print("전체 텍스트 비교 결과")
print("="*60)
print(f"Character Error Rate (CER): {metrics['CER']:.4f} ({metrics['CER']*100:.2f}%)")
print(f"Character Accuracy: {metrics['Character_Accuracy']:.2f}%")
print(f"\nWord Error Rate (WER): {metrics['WER']:.4f} ({metrics['WER']*100:.2f}%)")
print(f"Word Accuracy: {metrics['Word_Accuracy']:.2f}%")
print(f"\nSequence Similarity: {metrics['Sequence_Similarity']:.2f}%")
print(f"\nReference (Human) Length: {metrics['Reference_Length']} characters")
print(f"Hypothesis (Whisper) Length: {metrics['Hypothesis_Length']} characters")
print(f"Length Ratio: {metrics['Length_Ratio']:.2f}%")

## 8. 음소(Phoneme) 단위 비교 - G2P 변환

In [25]:
# G2P (Grapheme to Phoneme) 변환기 초기화
g2p = G2p()

def text_to_phonemes(text):
    """텍스트를 음소로 변환"""
    # 공백 제거 후 음소 변환
    text_no_space = text.replace(' ', '')
    phonemes = g2p(text_no_space)
    return phonemes

# 음소 변환
human_phonemes = text_to_phonemes(human_clean)
whisper_phonemes = text_to_phonemes(whisper_clean)

print("=== 사람 타이핑 음소 변환 (앞부분) ===")
print(human_phonemes[:200])
print("\n=== Whisper 음소 변환 (앞부분) ===")
print(whisper_phonemes[:200])

=== 사람 타이핑 음소 변환 (앞부분) ===
어어지베돌조라써요아으느미안해요제가시지하주차장에서시허가야케서모태써여괜차나요제자기소개저는하구이천시비니언사뭘달한국뽀러으로와써여네네한시보년정도돼써요그저는따리뚱이이써요큰떨마ㄱ내딸차이이써서요어큰떨른지그미쓰뭉요소쌀ㅁ앙내따르녈세살차이가조금마나써요지그는주부에요그냥지베서망내하꾜꽁부는거봐주구지베서뭐주부는거이리가하고이써서요응제가따릉그는한구거나이가조그미써서이허리도안조아서곤부는거는

=== Whisper 음소 변환 (앞부분) ===
소늘지베도라써요미안해요제가가사치아트타자에서씨너가야캐서모탣쬬괜차나요제자기소개저는한구기쟨시비년삼몰딸한구그로와써요거안쉬우녜정도돼써요그저는따리똥이이써요그날마네달차이이써써요캔다른지금스무려섣싸럼미더느녈세살차이가조금마나써요그러면지그믄주부예요그냥지베서망내학꾜공부는거봐주고네지베서뭐주부는거일회가하고이써써요제가따른나는한구거나이가조그미써서리얼리도안조아서공부는하구공부는지그만가요다


In [ ]:
# 음소 단위 비교
phoneme_metrics = calculate_similarity_metrics(human_phonemes, whisper_phonemes)

print("="*60)
print("음소(Phoneme) 단위 비교 결과")
print("="*60)
print(f"Phoneme Error Rate (PER): {phoneme_metrics['CER']:.4f} ({phoneme_metrics['CER']*100:.2f}%)")
print(f"Phoneme Accuracy: {phoneme_metrics['Character_Accuracy']:.2f}%")
print(f"\nPhoneme Sequence Similarity: {phoneme_metrics['Sequence_Similarity']:.2f}%")

## 9. 문장별 비교 분석

In [27]:
# 사람 타이핑 텍스트를 줄 단위로 분리
human_lines = [line.strip() for line in human_transcript.split('\n') if line.strip()]

print(f"사람 타이핑 텍스트 줄 수: {len(human_lines)}")
print("\n=== 처음 10줄 ===")
for i, line in enumerate(human_lines[:10]):
    print(f"{i+1}: {line[:80]}..." if len(line) > 80 else f"{i+1}: {line}")

사람 타이핑 텍스트 줄 수: 24

=== 처음 10줄 ===
1: 어어 집에 돌(졸)았어요. 아으느 미안해요 제가 시(지)하 주차장에서 시허가 야케서 모태써여
2: 괜차나요 제 자기소개 저는 하구 이천십이니언 삼월달 한국 보러(으로) 와써여
3: 네네 한 시보년 정도 돼써요
4: 그 저는 딸이 뚱이 이써요 큰 떨 마ㄱ내 딸 차이 이써서요 어 큰 떨는 지금 이 쓰뭉 요소쌀? ㅁ앙내 따른 열세살 차이가 조금 마나써요
5: 지그는 주부에요. 그냥 집에서 망내 하꾜 꽁부는 거 봐주구 집에서 뭐 주부는거 이리가 하고 이써서요
6: 응 제가 따릉그는 한구거 나이가 조금 있어서 이허리도 안 좋아서 곤부는 거는 한구 곤부는 지금 안 가요
7: 어 타릉거 남펴니 한구 사라미에요
8: 요즈믄 지자는 어려서 그 췌딸 팔딸 그냥 물 씅수 파에 씅 사라미 후빵 파에 씅 사라미 말 드러써요?
9: 후바 발씅 사라미에요
10: 어 따른 부는 또 다른 사라미 다 또가타요 그냥 보통 사라미에요


In [28]:
def find_best_match(target, segments_list, threshold=0.3):
    """주어진 텍스트와 가장 유사한 세그먼트 찾기"""
    target_clean = preprocess_text(target)
    if not target_clean:
        return None, 0
    
    best_match = None
    best_score = 0
    
    for seg in segments_list:
        seg_clean = preprocess_text(seg['text'])
        if not seg_clean:
            continue
        
        score = SequenceMatcher(None, target_clean, seg_clean).ratio()
        if score > best_score:
            best_score = score
            best_match = seg
    
    if best_score < threshold:
        return None, best_score
    
    return best_match, best_score

# 각 줄별로 매칭 찾기 (처음 20줄만)
print("=== 줄별 매칭 분석 (처음 20줄) ===")
print("="*80)

for i, line in enumerate(human_lines[:20]):
    match, score = find_best_match(line, segments_data)
    print(f"\n[줄 {i+1}] 사람 타이핑:")
    print(f"  {line[:60]}..." if len(line) > 60 else f"  {line}")
    if match:
        print(f"  → Whisper 매칭 ({score*100:.1f}% 유사도):")
        print(f"     {match['text'][:60]}..." if len(match['text']) > 60 else f"     {match['text']}")
    else:
        print(f"  → 매칭 없음 (최고 유사도: {score*100:.1f}%)")

=== 줄별 매칭 분석 (처음 20줄) ===

[줄 1] 사람 타이핑:
  어어 집에 돌(졸)았어요. 아으느 미안해요 제가 시(지)하 주차장에서 시허가 야케서 모태써여
  → Whisper 매칭 (50.0% 유사도):
     손을 집에 돌았어요. 미안해요. 제가

[줄 2] 사람 타이핑:
  괜차나요 제 자기소개 저는 하구 이천십이니언 삼월달 한국 보러(으로) 와써여
  → Whisper 매칭 (44.4% 유사도):
     괜찮아요 제 자기소개 저는

[줄 3] 사람 타이핑:
  네네 한 시보년 정도 돼써요
  → Whisper 매칭 (48.3% 유사도):
     거 안 쉬운 예정도 됐어요.

[줄 4] 사람 타이핑:
  그 저는 딸이 뚱이 이써요 큰 떨 마ㄱ내 딸 차이 이써서요 어 큰 떨는 지금 이 쓰뭉 요소쌀? ㅁ앙내 따른 ...
  → 매칭 없음 (최고 유사도: 25.5%)

[줄 5] 사람 타이핑:
  지그는 주부에요. 그냥 집에서 망내 하꾜 꽁부는 거 봐주구 집에서 뭐 주부는거 이리가 하고 이써서요
  → Whisper 매칭 (48.7% 유사도):
     그냥 집에서 막내 학교 공부는 거 봐주고 네

[줄 6] 사람 타이핑:
  응 제가 따릉그는 한구거 나이가 조금 있어서 이허리도 안 좋아서 곤부는 거는 한구 곤부는 지금 안 가요
  → Whisper 매칭 (45.6% 유사도):
     제가 따른나는 한국어 나이가 조금 있어서

[줄 7] 사람 타이핑:
  어 타릉거 남펴니 한구 사라미에요
  → Whisper 매칭 (51.4% 유사도):
     다른 그 남편이 한국 사람이에요

[줄 8] 사람 타이핑:
  요즈믄 지자는 어려서 그 췌딸 팔딸 그냥 물 씅수 파에 씅 사라미 후빵 파에 씅 사라미 말 드러써요?
  → 매칭 없음 (최고 유사도: 26.7%)

[줄 9] 사람 타이핑:
  후바 발씅 사라미에요
  → Whisper 매칭 (45.5% 유사도):
     제가 중국 사람이에요.

[줄 10] 사람 타이핑:
  어 따른 부는 또 다른

## 10. 세부 분석 - 특정 패턴 비교

In [29]:
def analyze_pronunciation_patterns(human_text, whisper_text):
    """
    발음 패턴 분석: 사람이 들은 대로 적은 텍스트와 Whisper 인식 비교
    - 어눌한 발음이 어떻게 다르게 인식되는지 분석
    """
    # 자주 나타나는 발음 패턴 (인터뷰 타이핑 파일 기반)
    pronunciation_patterns = {
        '어눌한 발음 → 표준 발음 추정': [
            ('졸', '도착'),  # 돌(졸)았어요 → 도착
            ('시허', '신호'),  # 시허가 → 신호가
            ('야케', '약해'),  # 야케서 → 약해서
            ('모태', '못해'),  # 모태써여 → 못했어요
            ('마나', '많아'),  # 마나써요 → 많았어요
            ('쪼끔', '조금'),  # 쪼끔 → 조금
            ('구거', '국어'),  # 구거 → 국어
            ('수하', '수학'),  # 수하 → 수학
            ('꽁부', '공부'),  # 꽁부 → 공부
            ('하꾜', '학교'),  # 하꾜 → 학교
            ('쭝구', '중국'),  # 쭝구 → 중국
            ('신드러', '힘들어'),  # 신드러요 → 힘들어요
            ('평원', '병원'),  # 평원 → 병원
            ('짜신감', '자신감'),  # 짜신감 → 자신감
        ]
    }
    
    print("=== 발음 패턴 분석 ===")
    print("사람이 들은 발음 → Whisper가 인식한 텍스트")
    print("-"*60)
    
    for category, patterns in pronunciation_patterns.items():
        print(f"\n{category}:")
        for heard, standard in patterns:
            # 사람 타이핑에서 해당 패턴 출현 횟수
            human_count = human_text.lower().count(heard)
            # Whisper에서 표준 발음 출현 횟수
            whisper_standard_count = whisper_text.count(standard)
            whisper_heard_count = whisper_text.lower().count(heard)
            
            if human_count > 0:
                print(f"  '{heard}' (사람: {human_count}회) → "
                      f"Whisper: '{standard}'({whisper_standard_count}회), "
                      f"'{heard}'({whisper_heard_count}회)")

analyze_pronunciation_patterns(human_transcript, whisper_transcript)

=== 발음 패턴 분석 ===
사람이 들은 발음 → Whisper가 인식한 텍스트
------------------------------------------------------------

어눌한 발음 → 표준 발음 추정:
  '졸' (사람: 1회) → Whisper: '도착'(0회), '졸'(0회)
  '시허' (사람: 1회) → Whisper: '신호'(0회), '시허'(0회)
  '야케' (사람: 1회) → Whisper: '약해'(1회), '야케'(0회)
  '모태' (사람: 1회) → Whisper: '못해'(1회), '모태'(0회)
  '마나' (사람: 3회) → Whisper: '많아'(1회), '마나'(0회)
  '쪼끔' (사람: 1회) → Whisper: '조금'(9회), '쪼끔'(0회)
  '구거' (사람: 9회) → Whisper: '국어'(6회), '구거'(0회)
  '수하' (사람: 2회) → Whisper: '수학'(0회), '수하'(1회)
  '꽁부' (사람: 4회) → Whisper: '공부'(10회), '꽁부'(0회)
  '하꾜' (사람: 3회) → Whisper: '학교'(2회), '하꾜'(0회)
  '쭝구' (사람: 3회) → Whisper: '중국'(1회), '쭝구'(0회)
  '신드러' (사람: 1회) → Whisper: '힘들어'(0회), '신드러'(0회)


## 11. 결과 요약 및 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (필요시)
# plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. 일치율 비교 차트
categories = ['Character\nAccuracy', 'Word\nAccuracy', 'Sequence\nSimilarity', 'Phoneme\nAccuracy']
values = [
    metrics['Character_Accuracy'],
    metrics['Word_Accuracy'],
    metrics['Sequence_Similarity'],
    phoneme_metrics['Character_Accuracy']
]

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']
bars = axes[0].bar(categories, values, color=colors)
axes[0].set_ylim(0, 100)
axes[0].set_ylabel('Accuracy / Similarity (%)')
axes[0].set_title('Human Transcript vs Whisper: Similarity Metrics')

# 값 표시
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

# 2. 오류율 비교 차트
error_categories = ['CER', 'WER', 'PER']
error_values = [
    metrics['CER'] * 100,
    metrics['WER'] * 100,
    phoneme_metrics['CER'] * 100
]

error_colors = ['#e74c3c', '#f39c12', '#8e44ad']
bars2 = axes[1].bar(error_categories, error_values, color=error_colors)
axes[1].set_ylabel('Error Rate (%)')
axes[1].set_title('Error Rates (Lower is Better)')

for bar, val in zip(bars2, error_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'comparison_results.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n차트 저장 완료: {os.path.join(WORK_DIR, 'comparison_results.png')}")

## 12. 최종 결과 저장

In [ ]:
# 최종 결과 저장
final_results = {
    "audio_file": AUDIO_PATH,
    "transcript_file": TRANSCRIPT_PATH,
    "g2p_ft_model": "whisper-small + LoRA fine-tuned",  # 🔥 model_name → 직접 문자열
    "finetuned_model_path": "data/들리는json/no7_model_whisper-g2p-final",  # 🔥 추가
    "metrics": {
        "character_error_rate": metrics['CER'],
        "character_accuracy": metrics['Character_Accuracy'],
        "word_error_rate": metrics['WER'],
        "word_accuracy": metrics['Word_Accuracy'],
        "sequence_similarity": metrics['Sequence_Similarity'],
        "phoneme_error_rate": phoneme_metrics['CER'],
        "phoneme_accuracy": phoneme_metrics['Character_Accuracy'],
    },
    "text_lengths": {
        "human_transcript": len(human_clean),
        "whisper_transcript": len(whisper_clean)
    },
    "whisper_full_text": whisper_transcript,
    "total_segments": len(segments_data)
}

results_file = os.path.join(WORK_DIR, 'analysis_results.json')
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"결과 저장 완료: {results_file}")

In [ ]:
# Whisper 변환 텍스트 별도 저장
whisper_text_file = os.path.join(WORK_DIR, 'whisper_transcript.txt')
with open(whisper_text_file, 'w', encoding='utf-8') as f:
    f.write(whisper_transcript)

print(f"Whisper 변환 텍스트 저장: {whisper_text_file}")

## 13. 분석 결과 해석

### 주요 지표 설명:

1. **CER (Character Error Rate)**: 문자 단위 오류율. 삽입, 삭제, 대체 오류의 총합을 참조 텍스트 길이로 나눈 값.
   - 낮을수록 좋음
   - 100% 이상이면 Whisper가 완전히 다른 텍스트를 생성한 것

2. **WER (Word Error Rate)**: 단어 단위 오류율. 띄어쓰기 기준으로 단어를 분리하여 계산.
   - 한국어의 경우 형태소 분석이 필요할 수 있음

3. **Sequence Similarity**: 전체 시퀀스의 유사도. difflib의 SequenceMatcher 사용.
   - 0-100% 범위, 높을수록 유사함

4. **PER (Phoneme Error Rate)**: 음소 단위 오류율. G2P로 변환 후 비교.
   - 발음 유사도를 더 정확하게 측정
   - 표기가 달라도 발음이 같으면 일치로 처리

### 해석 시 고려사항:

- 사람 타이핑 텍스트는 **들리는 대로** 적은 것 (어눌한 발음 그대로)
- Whisper는 **표준 한국어**로 변환하려는 경향이 있음
- 따라서 낮은 일치율은 예상되는 결과임
- 중요한 것은 Whisper가 화자의 **의도**를 얼마나 정확히 파악했는가

In [ ]:
# 최종 요약 출력
print("=" * 70)
print("              Fine-tuned 모델 분석 결과 최종 요약")
print("=" * 70)
print(f"\n사용 모델: Whisper-small + LoRA Fine-tuned")
print(f"Fine-tuned 모델 경로: {FINETUNED_MODEL_PATH}")
print(f"\n텍스트 길이:")
print(f"  - 사람 타이핑: {len(human_clean):,} 문자")
print(f"  - Fine-tuned Whisper 변환: {len(whisper_clean):,} 문자")
print(f"\n일치율 분석:")
print(f"  - 문자 정확도 (Character Accuracy): {metrics['Character_Accuracy']:.2f}%")
print(f"  - 단어 정확도 (Word Accuracy): {metrics['Word_Accuracy']:.2f}%")
print(f"  - 시퀀스 유사도 (Sequence Similarity): {metrics['Sequence_Similarity']:.2f}%")
print(f"  - 음소 정확도 (Phoneme Accuracy): {phoneme_metrics['Character_Accuracy']:.2f}%")
print(f"\n오류율:")
print(f"  - CER: {metrics['CER']*100:.2f}%")
print(f"  - WER: {metrics['WER']*100:.2f}%")
print(f"  - PER: {phoneme_metrics['CER']*100:.2f}%")
print("\n" + "=" * 70)
print("분석 완료!")
print("=" * 70)